In [1]:
# リカレントニューラルネットワーク（RNN）
# 確率と言語モデル
# ord2vecを確率の視点から眺める
# 言語モデル
# CBOWモデルを言語モデルに?

In [2]:
# RNNとは
# 循環するニューラルネットワーク
# ループの展開
# Backpropagation Through Time
# Trucated BPTT
# Truncated BPTTのミニバッチ学習

In [ ]:
# RNNの実装
# RNNレイヤの実装
import numpy as np

class RNN:
    def __init__(self, Wx, Wh, b):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None

    def forward(self, x, h_prev):
        Wx, Wh, b = self.params
        t = np.dot(h_prev, Wh) + np.dot(x, Wx) + b
        h_next = np.tanh(t)

        self.cache = (x, h_prev, h_next)
        return h_next

    def backward(self, dh_next):
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache

        # dt：順伝播の最後のtanhゲートの逆伝播
        # 上流から流れてきた勾配 dh_next × tanhの微分
        dt = dh_next * (1 - h_next ** 2)
        db = np.sum(dt, axis=0)
        # dWh：順伝播でWhと一緒に掛けたものh_prevと上流からの勾配を掛ける
        dWh = np.dot(h_prev.T, dt)
        # dh_prev：順伝播でh_prevと一緒に掛けたものWhと上流からの勾配を掛ける
        dh_prev = np.dot(dt, Wh.T)
        dWx = np.dot(x.T, dt)
        dx = np.dot(dt, Wx.T)

        # [...]：配列の形やメモリの位置はそのままキープして中身のデータだけ入れ替える
        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev

In [ ]:
# TimeRNNレイヤの実装
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateful=False):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None

        self.h, self.dh = None, None
        self.stateful = stateful

    def forward(self, xs):
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        D, H = Wx.shape

        # 1ステップ分のRNNレイヤをT個作って保管するためのリスト
        # backwardでこのリストを後ろから回すときに使う
        self.layers = []
        hs = np.empty((N, T, H), dtype='f')

        # Truncated BPTTの仕組み
        # True：記憶は引き継がない（0行列でリセット）
        # False：記憶を引き継ぐ（self.hを使いまわす）
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')

        # 時間方向（T）のループ
        for t in range(T):
            # 1ステップ分のRNNレイヤを作る
            # (*self.params)：共通の重みをバラして渡す
            layer = RNN(*self.params)
            # xs[:, t, :]：3次元データxsから、時刻tのデータだけをスライスして取り出す（N, D）
            # 今作ったRNNレイヤに、その時刻tの入力xsと、これまでの記憶self.hを入れて、
            # 新しい記憶をself.hに上書きする
            self.h = layer.forward(xs[:, t, :], self.h)
            # さっき用意したhsの箱の時刻tの場所に、self.hを保存する
            hs[:, t, :] = self.h
            # self.layersにRNNレイヤを追加
            self.layers.append(layer)

        # 全ての時刻が埋まった記憶hsを返す（N, T, H）
        return hs

    def backward(self, dhs):
        Wx, Wh, b = self.params
        # 順伝播と同じく、入ってきたデータの形状の確認をする
        N, T, H = dhs.shape
        D, H = Wx.shape

        dxs = np.empty((N, T, D), dtype='f')
        # 未来のブロックから伝わってくる隠れ状態の勾配を入れる変数
        # 最初は0で初期化
        dh = 0
        # 全時刻の重みの勾配を足し算して溜めていくリスト
        grads = [0, 0, 0]
        # 未来から過去へ、時刻Tを逆伝播
        for t in reversed(range(T)):
            # forwardで保存しておいたRNNレイヤを1つずつ取り出す
            layer = self.layers[t]
            # dhs[:, t, :]：その時刻t、出力側から回ってきた勾配
            # dh：1つ未来の時刻t + 1から伝わってきた勾配
            # ➡ 2つの勾配を足してRNNレイヤに渡す
            dx, dh = layer.backward(dhs[:, t, :] + dh)
            # dxs：入力データに対する勾配
            dxs[:, t, :] = dx

            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        for i, grad in enumerate(grads):
            self.grads[i][...] = grad
        self.dh = dh

        return dxs
